# boolean-mask-identity-replace — ex8: outlier removal with combined boolean masks

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `boolean-mask-identity-replace`. Running the final beacon cell reports progress against the `Numpy: Indexing and selection` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Numpy: Indexing and selection` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`boolean-mask-identity-replace`** (exercise 8). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "boolean-mask-identity-replace"
DD_SUBTOPIC = "Numpy: Indexing and selection"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Mask & substitute — quick refresher

**Build a mask.** Any comparison returns a `dtype=bool` tensor of the same shape: `x < 0`, `x.abs() < eps`, `(x > 0) & (x < 1)`. Combine with `&`, `|`, `~`.

**Write through a mask.** `y[mask] = value` modifies in place. Scalars broadcast; tensor values must match the shape of `y[mask]` after broadcasting. Always `clone()` first if the function must not mutate its input.

**The dangerous case.** When a mask is the *wrong* shape, indexing can silently collapse axes or pick the wrong cells. Always check `mask.sum()` and `mask.shape` before trusting the result.

### Exercise 8 — outlier removal with combined boolean masks

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Combine 2-3 boolean masks via &/|/~ to select a subset of rows, and visualize the keep/drop split.
> Keywords: mask-combine, bitwise-and, bitwise-or, scatter-plot, outlier
> ```

**KCs targeted:** `combine-masks-with-&-and-|`, `negate-mask-with-~`, `boolean-row-selection`

Given a `(N, 2)` array of 2-D points `pts` and an `(N,)` quality score `score`, implement `ex8_keep_clean_points(pts, score)` to return three things:

- `kept` — the subset of `pts` that satisfy ALL of:
  1. inside the unit box: `-1 ≤ x ≤ 1` AND `-1 ≤ y ≤ 1`
  2. `score > 0.5`
  3. NOT (both coordinates near zero): NOT (`abs(x) < 0.05` AND `abs(y) < 0.05`)  ← rejects degenerate origin cluster
- `kept_mask` — the `(N,)` bool mask used to select `kept`
- `dropped_mask` — the negation, `~kept_mask`

Return as a tuple `(kept, kept_mask, dropped_mask)`. Build each criterion as its own bool tensor first, then combine with `&` and `~`. The test cell will use the masks to color points kept vs dropped on a scatter plot — you should see only the 'good' annulus survive.

In [ ]:
def ex8_keep_clean_points(pts: Tensor, score: Tensor):
    """Filter (N, 2) points by combined criteria.

    Returns (kept, kept_mask, dropped_mask).
    """
    raise NotImplementedError()


def _test_ex8():
    # Hand-crafted set of 8 points covering each criterion
    pts = t.tensor([
        [0.0, 0.0],     # 0  origin → drop (criterion 3)
        [0.5, 0.5],     # 1  inside, score>0.5, not origin → keep
        [2.0, 0.0],     # 2  outside unit box → drop (criterion 1)
        [-0.5, 0.5],    # 3  inside, score>0.5, not origin → keep
        [0.5, -0.5],    # 4  inside, score=0.3 (below thresh) → drop (criterion 2)
        [0.01, 0.01],   # 5  near origin → drop (criterion 3)
        [0.9, 0.9],     # 6  inside, score>0.5, not origin → keep
        [-1.5, 0.0],    # 7  outside unit box → drop
    ])
    score = t.tensor([0.9, 0.9, 0.9, 0.9, 0.3, 0.9, 0.9, 0.9])
    kept, km, dm = ex8_keep_clean_points(pts, score)
    expected_keep_idx = t.tensor([False, True, False, True, False, False, True, False])
    assert t.equal(km, expected_keep_idx), f'kept_mask mismatch: {km.tolist()}'
    assert t.equal(dm, ~expected_keep_idx), 'dropped_mask must be negation of kept_mask'
    assert kept.shape == (3, 2), f'expected 3 kept points, got {tuple(kept.shape)}'
    assert t.equal(kept, pts[expected_keep_idx]), 'kept rows must equal pts[kept_mask]'

    # Cardinality
    assert km.sum().item() == 3
    assert dm.sum().item() == 5
    assert (km & dm).sum().item() == 0, 'kept and dropped must be disjoint'
    assert (km | dm).sum().item() == km.numel(), 'kept | dropped must cover all rows'

    # Visualize on a larger random cloud
    import matplotlib.pyplot as plt
    t.manual_seed(42)
    N = 400
    pts_big = t.empty(N, 2).uniform_(-1.5, 1.5)
    score_big = t.rand(N)
    kp, kmb, dmb = ex8_keep_clean_points(pts_big, score_big)
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.scatter(pts_big[dmb, 0].numpy(), pts_big[dmb, 1].numpy(),
               c='lightgray', s=12, label=f'dropped ({dmb.sum().item()})')
    ax.scatter(pts_big[kmb, 0].numpy(), pts_big[kmb, 1].numpy(),
               c='tab:blue', s=18, label=f'kept ({kmb.sum().item()})')
    ax.axhline(0, lw=0.3); ax.axvline(0, lw=0.3)
    ax.set_aspect('equal'); ax.set_xlim(-1.6, 1.6); ax.set_ylim(-1.6, 1.6)
    ax.set_title('outlier removal — kept vs dropped')
    ax.legend(loc='upper right')
    plt.tight_layout(); plt.show()
    _dd_passed.add('ex8')
    print("ex8 ✓")

_test_ex8()

<details><summary>Solution</summary>

```python
def ex8_keep_clean_points(pts: Tensor, score: Tensor):
    x, y = pts[:, 0], pts[:, 1]
    in_box       = (x >= -1) & (x <= 1) & (y >= -1) & (y <= 1)
    high_score   = score > 0.5
    near_origin  = (x.abs() < 0.05) & (y.abs() < 0.05)
    kept_mask    = in_box & high_score & (~near_origin)
    dropped_mask = ~kept_mask
    return pts[kept_mask], kept_mask, dropped_mask
```

**`&` vs `and`.** Always use `&` / `|` / `~` for bool tensors. Python `and` / `or` short-circuit on a scalar bool, so on a tensor they raise `RuntimeError: Boolean value of Tensor with more than one element is ambiguous`. The bitwise operators are elementwise.

**Parens matter.** `x >= -1 & x <= 1` parses as `x >= (-1 & x) <= 1` because `&` binds tighter than `>=`. Always parenthesize each comparison: `(x >= -1) & (x <= 1)`.

**`pts[mask]` vs `pts[mask, :]`.** When `mask` is 1-D, both are equivalent and produce a 2-D result. When the mask matches multiple axes (e.g. 2-D mask on 2-D tensor), the result is 1-D and you've collapsed the geometry — that's a common surprise.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex8'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex8',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()